In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
from pybaseball import standings
from pybaseball import team_batting

Eventually I import the different files and so the user gets to choose
df.head()
df.columns
df.info()
basic panda functions to use asap for new things

This is my new project to learn more about Jupyter Notebooks, python, and SQL

teams          → pybaseball standings data
team_records   → cleaned standings DataFrame
teams_api      → raw MLB API team data

In [ ]:
teams = standings(2025)

In [ ]:
teams[0]

In [ ]:
df = pd.concat(teams, ignore_index=True)
df
df.shape

In [ ]:
#looking at the exact cols in the database
df.head

In [ ]:
df.columns

In [ ]:
team_records = df[["Tm", "W", "L", "W-L%","GB"]].copy()
team_records.head()
team_win_rate = df[["W-L%"]]
team_win_rate

In [ ]:
team_records.sort_values(by="W")

In [ ]:
team_records.dtypes

In [ ]:
team_records[["W", "L", "W-L%"]] = team_records[["W", "L", "W-L%"]].apply(pd.to_numeric)


In [ ]:
team_records.dtypes

In [ ]:
team_records[team_records["W"] >= 90]

In [ ]:
type(team_records["W"].iloc[0])

In [ ]:
team_records[team_records["W-L%"] >= 90]

In [ ]:
team_Tm_W = df[["Tm", "W"]]
team_Tm_W

In [ ]:
df.compare

In [ ]:
df.info()

In [ ]:
team_records["Win_Percentage"] = team_records["W"] / (team_records["W"] + team_records["L"])

team_records

Here is stuff from batting now


In [ ]:
#batting = team_batting(2025, league="all")

In [ ]:
#batting.head()
#batting.shape
#batting.columns
#batting.info()

Here is stuff for the 2024 season will eventually compare stuff

In [ ]:
teams2024 = standings(2024)
df2024 = pd.concat(teams2024, ignore_index=True)
df2024
df2024.shape


In [ ]:
df2024.columns

In [ ]:
df2024.head

In [ ]:
df2024.compare

In [ ]:
df2024.info()

## Different api
This is gonna be another api to pull data from to learn more about the team

In [ ]:
%pip install MLB-StatsAPI

In [ ]:
import statsapi


In [ ]:
teams_api = statsapi.get("teams", {"sportId": 1})

In [ ]:
teams_api["teams"][0]
teams_api

In [ ]:
type(teams_api)

In [ ]:
type(teams_api["teams"])
len(teams_api["teams"])

Making Teams_api into a dataframe to make it look good

In [ ]:
api_teams_df = pd.DataFrame(teams_api["teams"])

In [ ]:
api_teams_df.head()

In [ ]:
api_teams_df.shape

In [ ]:
api_teams_df.columns

In [ ]:
api_teams_df.info()

Add a column teams_info for just the name win percentages and division

In [ ]:
teams_info = api_teams_df[["name", "season", "division"]].copy()
teams_info

In [ ]:
teams_info.head(8)

In [ ]:
teams_info["division"].iloc[0]

This shows how dirty and hard to read the dictionary inside of division is
A person really only needs to see the "name" portion of the dictionary      

In [ ]:
div = teams_info["division"]
div.iloc[0]

In [ ]:
# div.iloc[0,29]['name']
division = teams_info["division"].str["name"]
division

In [ ]:
teams_info["Division Str"] = division
teams_info.head(8)

In [ ]:
teams_info = teams_info[["name", "season", "Division Str"]]
teams_info


In [ ]:
data_dir = Path.cwd() / "data"
if not data_dir.exists():
    data_dir = Path.cwd().parent / "data"
connection = sqlite3.connect(data_dir / "mlb.db")

In [ ]:
team_records.to_sql(
    "team_records",
    connection,
    if_exists="replace",
    index=False
)

In [ ]:
query = """
SELECT *
FROM team_records;
"""

pd.read_sql_query(query, connection)

In [ ]:
# add rate_team from src/main

In [ ]:
print(rate_team(0.610))
print(rate_team(0.570))
print(rate_team(0.520))
print(rate_team(0.420))

In [ ]:
connection.execute(
      "ALTER TABLE team_records ADD COLUMN Season INTEGER"
  )

  connection.execute(
      "UPDATE team_records SET Season = 2025"
  )

  connection.commit()